In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.train_temporal import CV_SUMMARY_PATH
from src.train_baselines import BASELINE_SUMMARY_PATH

gru_summary = pd.read_csv(CV_SUMMARY_PATH)
gru_summary = gru_summary.rename(columns={"mean": "gru_mean", "std": "gru_std"})

logreg_summary = pd.read_csv(BASELINE_SUMMARY_PATH)
logreg_summary = logreg_summary.rename(columns={"mean": "logreg_mean", "std": "logreg_std"})

comparison_df = gru_summary.merge(logreg_summary, on="metric", how="inner")
comparison_df

,metric,gru_mean,gru_std,logreg_mean,logreg_std
0,accuracy,0.500000,0.204124,0.633333,0.074536
1,precision,0.390000,0.260768,0.673333,0.192065
2,recall,0.600000,0.434613,0.666667,0.333333
3,f1,0.471429,0.324784,0.613333,0.156968


This table compares the primary temporal GRU model against a static logistic regression baseline trained on mean-pooled clip-level pose features.

In [2]:
report_comparison = comparison_df.copy()

report_comparison["GRU"] = report_comparison.apply(
    lambda row: f"{row['gru_mean']:.3f} ± {row['gru_std']:.3f}",
    axis=1
)

report_comparison["Logistic Regression"] = report_comparison.apply(
    lambda row: f"{row['logreg_mean']:.3f} ± {row['logreg_std']:.3f}",
    axis=1
)

metric_name_map = {
    "val_loss": "Validation Loss",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1 Score",
}

report_comparison["metric"] = report_comparison["metric"].map(metric_name_map).fillna(report_comparison["metric"])

report_comparison = report_comparison[["metric", "GRU", "Logistic Regression"]]
report_comparison

,metric,GRU,Logistic Regression
0,Accuracy,0.500 ± 0.204,0.633 ± 0.075
1,Precision,0.390 ± 0.261,0.673 ± 0.192
2,Recall,0.600 ± 0.435,0.667 ± 0.333
3,F1 Score,0.471 ± 0.325,0.613 ± 0.157


In [3]:
comparison_path = PROJECT_ROOT / "data" / "processed" / "model_comparison.csv"
report_comparison.to_csv(comparison_path, index=False)

print(comparison_path)
print(comparison_path.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/model_comparison.csv
True


Under the current settings and small-data regime, logistic regression outperforms the GRU on preliminary cross-validated metrics.